LOAD FILE

In [1]:
import pandas as pd
df = pd.read_excel("DataEntry.xlsx")
df.head()

,population,income_avg,education_index,health_access_score,risk_label
0,16795,7.950203e+06,0.706539,88.561669,0
1,1860,2.788441e+06,0.309953,88.606804,1
2,77820,1.049699e+06,0.607256,92.024339,1
3,55886,8.339153e+06,0.435897,94.794433,0
4,7265,7.361716e+06,0.687104,70.680544,0


In [2]:
df.describe()

,population,income_avg,education_index,health_access_score,risk_label
count,100.000000,1.000000e+02,100.000000,100.000000,100.000000
mean,52401.580000,5.306748e+06,0.617009,69.112689,0.440000
std,28762.687103,2.603282e+06,0.177254,17.069339,0.498888
min,1769.000000,1.049699e+06,0.303037,40.863609,0.000000
25%,26318.500000,3.151727e+06,0.458032,54.976895,0.000000
50%,57238.500000,5.424618e+06,0.641511,70.829282,0.000000
75%,75686.750000,7.624832e+06,0.752902,82.143377,1.000000
max,97276.000000,9.870854e+06,0.894032,98.551125,1.000000


In [3]:
df.columns.tolist()

['population',
 'income_avg',
 'education_index',
 'health_access_score',
 'risk_label']

AUTOMATD VALIDATION ERROR LOGGING

In [10]:
errors = []

def validate(row, idx):
    if row["population"] <= 0:
        errors.append((idx, "Invalid population"))
    if row["income_avg"] <= 0:
        errors.append((idx, "Invalid income"))
    if not (0 <= row["education_index"] <= 1):
        errors.append((idx, "Education index out of range"))
    if not (0 <= row["health_access_score"] <= 100):
        errors.append((idx, "Health score out of range"))
    if row["risk_label"] not in [0,1]:
        errors.append((idx, "Invalid risk label"))

for i, r in df.iterrows():
    validate(r, i)

error_log = pd.DataFrame(errors, columns=["row","error"])

error_log.to_csv("error_log.csv", index=False)

print("Validation done")
print("Total errors:", len(error_log))


Validation done
Total errors: 0


AUTOMATED CLEANING

In [12]:
df_clean = df.copy()

df_clean["education_index"] = df_clean["education_index"].clip(0,1)
df_clean["health_access_score"] = df_clean["health_access_score"].clip(0,100)
df_clean = df_clean.dropna()


print("cleaned rows:", len(df_clean))

cleaned rows: 100


In [13]:
df_clean.head(10)

,population,income_avg,education_index,health_access_score,risk_label
0,16795,7.950203e+06,0.706539,88.561669,0
1,1860,2.788441e+06,0.309953,88.606804,1
2,77820,1.049699e+06,0.607256,92.024339,1
3,55886,8.339153e+06,0.435897,94.794433,0
4,7265,7.361716e+06,0.687104,70.680544,0
5,83386,7.561065e+06,0.404620,70.090978,1
6,38194,7.941433e+06,0.714563,87.897711,0
7,88498,1.666402e+06,0.532041,78.997836,0
8,45131,4.226192e+06,0.862038,82.118013,0
9,61263,2.042822e+06,0.382513,87.747560,1


In [14]:
df_clean.tail()

,population,income_avg,education_index,health_access_score,risk_label
95,52214,8.517722e+06,0.744461,63.166158,1
96,62228,3.887021e+06,0.718209,97.671434,0
97,49984,2.678667e+06,0.721490,94.321039,0
98,41774,1.366976e+06,0.515695,51.747468,1
99,3568,6.318036e+06,0.476155,44.161678,0


ETL to MySQL

In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host="YOUR_HOST",
    user="YOUR_USER",
    password="YOUR_PASSWORD",
    database="YOUR_DATABASE"
)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS cleaned_data (
    population INT,
    income_avg FLOAT,
    education_index FLOAT,
    health_access_score FLOAT,
    risk_label INT
)
""")

for _, row in df_clean.iterrows():
    cursor.execute("""
        INSERT INTO cleaned_data VALUES (%s,%s,%s,%s,%s)
    """, tuple(row))

conn.commit()

print("ETL completed to MySQL")


DATA QUALITY ASSURANCE REPORT

In [15]:
quality_report = {
    "total_rows": len(df),
    "clean_rows": len(df_clean),
    "error_rows": len(error_log),
    "missing_values": df.isnull().sum().sum(),
    "duplicate_rows": df.duplicated().sum()
}

pd.DataFrame([quality_report])


,total_rows,clean_rows,error_rows,missing_values,duplicate_rows
0,100,100,0,0,0
